# ML3. Теоретические вопросы


## 1. What is leave-one-out? Provide limitations and strengths. / Что такое leave-one-out? Укажите ограничения и сильные стороны.

`leave-one-out` - это частный случай `cross-validation`, при котором количество фолдов равно количеству объектов в датасете.

Если в датасете есть `N` объектов, то модель обучается `N` раз. На каждой итерации:

1. один объект используется как test part;
2. остальные `N - 1` объектов используются как training part;
3. качество модели считается на оставленном объекте;
4. после всех итераций метрики усредняются.

То есть каждый объект ровно один раз оказывается в test part и `N - 1` раз участвует в training part.

### Strengths / Сильные стороны

* Почти все данные используются для обучения на каждой итерации, поэтому схема хорошо подходит для маленьких датасетов.
* Каждый объект используется для проверки качества, поэтому оценка учитывает все наблюдения.
* Нет случайности в разбиении, если порядок данных не влияет на модель: результат не зависит от случайного выбора train/test split.
* Метод позволяет получить максимально подробную оценку поведения модели на отдельных объектах.

### Limitations / Ограничения

* Метод очень дорогой по времени: нужно обучить модель `N` раз. Если объектов много или модель тяжелая, это становится практически неудобно.
* Оценка качества может иметь высокую дисперсию: test part на каждой итерации состоит всего из одного объекта, поэтому отдельные выбросы могут сильно влиять на результат.
* Метод плохо подходит для временных рядов, если простое исключение одного объекта нарушает временную структуру данных.
* При наличии групп в данных, например нескольких заявок одного клиента, обычный `leave-one-out` может привести к information leakage, если похожие или связанные объекты попадают одновременно в train и test.
* Для некоторых метрик, особенно зависящих от распределения классов в test part, один объект в test может быть недостаточен. Например, ROC-AUC нельзя корректно посчитать на одном объекте.

## 2. How do Grid Search, Randomized Grid Search, and Bayesian optimization work? / Как работают Grid Search, Randomized Grid Search и Bayesian optimization?

Все три подхода используются для `hyperparameter optimization`, то есть для поиска хорошей комбинации гиперпараметров модели. Гиперпараметры - это внешние параметры, которые модель не оптимизирует сама во время fitting. Например, для `ElasticNet` такими гиперпараметрами являются `alpha` и `l1_ratio`.

Обычно процесс устроен так:

1. задается пространство возможных значений гиперпараметров;
2. выбирается комбинация гиперпараметров;
3. модель обучается на training part;
4. качество измеряется на validation part или через cross-validation;
5. выбирается комбинация, давшая лучшую метрику.

### Grid Search / Поиск по сетке

`Grid Search` - это полный перебор заранее заданной сетки гиперпараметров.

Например, если мы хотим подобрать параметры для `ElasticNet`:

* `alpha = [0.001, 0.01, 0.1, 1.0]`;
* `l1_ratio = [0.1, 0.5, 0.9]`.

`Grid Search` обучит модель на всех комбинациях этих значений: `4 * 3 = 12` моделей.

Плюс `Grid Search` в том, что он систематически проверяет все заданные варианты. Минус - он быстро становится очень дорогим: если гиперпараметров много, число комбинаций растет очень быстро. Кроме того, `Grid Search` проверяет только те значения, которые заранее внесены в сетку, и может пропустить хорошее значение между точками сетки.

### Randomized Grid Search / Рандомизированный поиск по сетке

`Randomized Grid Search` выбирает не все комбинации, а случайное подмножество вариантов из заданного пространства.

Вместо полного перебора мы задаем, например, что хотим проверить только 30 случайных комбинаций. Значения гиперпараметров могут выбираться из списков или из распределений.

Плюсы `Randomized Grid Search`:

* работает быстрее, чем полный `Grid Search`, если пространство большое;
* может найти хорошие значения при ограниченном бюджете времени;
* удобен, когда не все гиперпараметры одинаково важны.

Минусы:

* результат зависит от случайности;
* нет гарантии, что будут проверены лучшие комбинации;
* для воспроизводимости нужно фиксировать `random_state`.

### Bayesian optimization / Байесовская оптимизация

`Bayesian optimization` - более умный подход к поиску гиперпараметров. Он не просто перебирает варианты, а строит вероятностную модель зависимости между гиперпараметрами и качеством модели.

Идея такая:

1. сначала проверяется несколько комбинаций гиперпараметров;
2. по полученным результатам строится приближенная модель функции качества;
3. алгоритм выбирает следующую комбинацию так, чтобы либо исследовать неизвестную область, либо уточнить область, где качество уже выглядит хорошим;
4. после новой проверки вероятностная модель обновляется;
5. процесс повторяется до достижения лимита итераций или времени.

Главное отличие от `Grid Search` и `Randomized Grid Search`: `Bayesian optimization` учитывает результаты предыдущих экспериментов и использует их для выбора следующих экспериментов.

Плюсы:

* часто находит хорошее решение за меньшее число запусков модели;
* полезен, когда обучение модели дорогое;
* умеет учитывать сложную зависимость качества от гиперпараметров.

Минусы:

* сложнее устроен;
* имеет собственные настройки;
* при очень шумной метрике или плохо заданном пространстве поиска может работать нестабильно.

В Python для `Bayesian optimization` часто используют библиотеки `hyperopt` и `optuna`.

## 3. Explain classification of feature selection methods / Объясните классификацию методов отбора признаков

`Feature selection` - это процесс выбора наиболее полезных признаков и удаления слабых, шумных, избыточных или вредных признаков.

Зачем нужен `Feature selection`:

* уменьшить размерность данных;
* ускорить обучение модели;
* снизить риск overfitting;
* убрать шумные признаки;
* сделать модель проще и интерпретируемее;
* иногда улучшить качество на validation/test data.

Методы `Feature selection` можно классифицировать по нескольким признакам.

### Supervised и unsupervised / Методы с учителем и без учителя

`Supervised` методы используют target variable. Они оценивают, насколько признак связан с целевой переменной.

Примеры: `Pearson`, `Chi2`, `Lasso`, `permutation importance`, методы на основе важности признаков в деревьях.

`Unsupervised` методы не используют target variable. Они смотрят только на структуру признаков.

Примеры:

* удаление признаков с большим количеством пропусков;
* удаление константных и почти константных признаков;
* удаление сильно коррелирующих между собой признаков;
* методы снижения размерности, например PCA, хотя PCA скорее создает новые признаки, чем выбирает исходные.

### Filters / Фильтры

`Filters` оценивают признаки отдельно от конкретной модели. Обычно для каждого признака считается статистический критерий или мера связи с target, после чего признаки ранжируются.

Примеры: `Pearson`, `Chi2`, mutual information, correlation threshold.

Плюсы `filters`:

* быстрые;
* простые;
* не требуют многократного обучения модели.

Минусы:

* могут не учитывать взаимодействия между признаками;
* могут выбирать признаки, которые отдельно полезны, но вместе дают дублирующую информацию;
* результат не всегда оптимален для конкретной модели.

### Wrappers / Оберточные методы

`Wrappers` подбирают признаки через обучение модели. Метод пробует разные наборы признаков, обучает модель и сравнивает качество.

Примеры:

* forward selection - начинаем без признаков и постепенно добавляем лучшие;
* backward elimination - начинаем со всех признаков и постепенно удаляем худшие;
* recursive feature elimination, RFE.

Плюсы `wrappers`:

* учитывают конкретную модель;
* могут учитывать взаимодействия признаков;
* часто дают хорошее качество.

Минусы:

* вычислительно дорогие;
* могут переобучаться на validation part;
* требуют аккуратной проверки через cross-validation.

### Embedded / Встроенные методы

`Embedded` методы выполняют отбор признаков внутри процесса обучения модели. То есть модель сама в процессе fitting определяет, какие признаки важны.

Примеры:

* `Lasso`;
* модели с L1-регуляризацией;
* feature importance в деревьях решений, Random Forest, Gradient Boosting;
* `Ridge` может уменьшать веса признаков, но обычно не зануляет их полностью.

Плюсы `embedded`:

* обычно быстрее, чем `wrappers`;
* учитывают конкретную модель;
* отбор признаков встроен в обучение.

Минусы:

* результат зависит от выбранной модели;
* важность признаков может быть нестабильной при сильной корреляции между признаками;
* некоторые методы требуют масштабирования признаков.

## How Pearson works / Как работает Pearson

`Pearson` обычно означает коэффициент корреляции Пирсона. Он измеряет линейную связь между числовым признаком и числовой target variable.

Коэффициент принимает значения от `-1` до `1`:

* `1` - сильная положительная линейная связь: чем больше признак, тем больше target;
* `-1` - сильная отрицательная линейная связь: чем больше признак, тем меньше target;
* `0` - линейной связи почти нет.

В `Feature selection` можно посчитать `Pearson` correlation между каждым признаком и target, затем выбрать признаки с наибольшим абсолютным значением корреляции.

Пример: если цена квартиры сильно растет вместе с площадью, то признак площади будет иметь высокую положительную корреляцию с price.

Ограничения `Pearson`:

* видит только линейную зависимость;
* чувствителен к выбросам;
* не подходит напрямую для категориальных признаков без кодирования;
* высокая корреляция не доказывает причинно-следственную связь;
* если два признака сильно коррелируют с target, они также могут дублировать друг друга.

## How Chi2 works / Как работает Chi2

`Chi2` - это критерий хи-квадрат. В `Feature selection` он часто используется для оценки связи между неотрицательными признаками и категориальной target variable.

Идея `Chi2`:

1. для каждого признака сравниваются наблюдаемые частоты и ожидаемые частоты;
2. если распределение признака заметно отличается между классами target, значит признак может быть полезен;
3. чем больше значение статистики `Chi2`, тем сильнее связь признака с target.

Например, если признак `CatsAllowed` встречается с разной частотой в объявлениях с `low`, `medium` и `high` interest_level, то `Chi2` может показать, что этот признак связан с target.

Важно:

* `Chi2` требует неотрицательных значений признаков;
* часто используется с count features, binary features или one-hot encoded categorical features;
* не подходит для признаков с отрицательными значениями;
* оценивает связь признака с target отдельно, без учета взаимодействия с другими признаками.

## How Lasso works / Как работает Lasso

`Lasso` - это линейная модель с L1-регуляризацией. Она минимизирует ошибку модели, но дополнительно штрафует модель за большие абсолютные значения коэффициентов.

Идея функции потерь:

`loss = model_error + alpha * sum(abs(weights))`

Где:

* `model_error` - ошибка модели на данных;
* `weights` - коэффициенты модели;
* `alpha` - гиперпараметр, который задает силу регуляризации.

Главная особенность `Lasso`: L1-регуляризация может занулять коэффициенты некоторых признаков. Если коэффициент признака стал равен `0`, значит модель фактически перестала использовать этот признак.

Поэтому `Lasso` можно использовать как `embedded` метод `Feature selection`:

1. нормализовать признаки;
2. обучить `Lasso`;
3. посмотреть на коэффициенты;
4. оставить признаки с ненулевыми или наибольшими по модулю коэффициентами.

Почему важно нормализовать признаки: L1-регуляризация зависит от масштаба коэффициентов. Если признаки измерены в разных масштабах, модель может некорректно штрафовать их веса.

Ограничения `Lasso`:

* при сильной корреляции признаков может выбрать один из похожих признаков почти случайно;
* работает как линейная модель, поэтому плохо ловит сложные нелинейные зависимости без дополнительных преобразований;
* результат зависит от `alpha`;
* требует аккуратного масштабирования признаков.

## What permutation significance is / Что такое permutation significance

В README используется термин `permutation importance`. В вопросе написано `permutation significance`; по смыслу здесь речь о том же подходе - оценке важности признака через перестановку его значений.

`permutation importance` показывает, насколько качество модели ухудшится, если разрушить связь одного признака с target.

Алгоритм:

1. обучаем модель обычным способом;
2. считаем базовую метрику качества на validation или test part;
3. выбираем один признак;
4. случайно перемешиваем значения этого признака между объектами;
5. остальные признаки оставляем без изменений;
6. снова считаем метрику качества;
7. если качество сильно упало, признак важен;
8. если качество почти не изменилось, признак, скорее всего, мало полезен для модели.

Пример: если после перемешивания `price` качество предсказания `interest_level` резко падает, значит `price` важен для модели.

Плюсы `permutation importance`:

* можно применять почти к любой обученной модели;
* оценивает важность признака через влияние на реальную метрику;
* хорошо подходит для сравнения признаков после обучения модели.

Минусы:

* требует дополнительных вычислений;
* результат может зависеть от случайного перемешивания;
* при сильной корреляции признаков важность может быть занижена: если один признак испортили, другой похожий признак все еще может нести ту же информацию;
* важно считать importance на validation/test part, а не на train, чтобы не получить слишком оптимистичную картину.

## What SHAP is / Что такое SHAP

`SHAP` расшифровывается как `SHapley Additive exPlanations`. Это метод объяснения предсказаний модели, основанный на значениях Шепли из теории игр.

Главная идея `SHAP`: предсказание модели можно разложить на вклад отдельных признаков.

Для одного объекта `SHAP` отвечает на вопрос: какие признаки и насколько сдвинули предсказание модели относительно базового уровня.

Упрощенно:

`prediction = base_value + shap_value_1 + shap_value_2 + ... + shap_value_n`

Где:

* `base_value` - среднее или базовое предсказание модели;
* `shap_value_i` - вклад i-го признака в конкретное предсказание;
* положительный `SHAP value` увеличивает предсказание;
* отрицательный `SHAP value` уменьшает предсказание.

`SHAP` полезен в двух режимах.

### Local explanation / Локальное объяснение

`SHAP` объясняет конкретное предсказание для одного объекта.

Например, модель предсказала высокий `interest_level` для объявления. `SHAP` может показать, что предсказание повысили признаки `Elevator`, `LaundryinUnit`, `FitnessCenter`, а снизил его высокий `price`.

### Global explanation / Глобальное объяснение

Если агрегировать `SHAP values` по многим объектам, можно понять, какие признаки в целом важны для модели.

Например, можно посчитать среднее абсолютное значение `SHAP value` для каждого признака и получить глобальный рейтинг feature importance.

### How SHAP differs from ordinary feature importance / Чем SHAP отличается от обычной feature importance

Обычная feature importance часто говорит только, что признак важен. `SHAP` дополнительно показывает направление и величину влияния признака на конкретное предсказание.

Например, `permutation importance` может сказать, что `price` важен. `SHAP` может показать, что для одного объявления высокий `price` снизил вероятность `high interest_level`, а для другого объекта влияние было другим из-за сочетания признаков.

### SHAP advantages / Плюсы SHAP

* дает интерпретируемые объяснения для отдельных предсказаний;
* позволяет строить глобальную важность признаков;
* показывает направление влияния признаков;
* применим к разным типам моделей;
* особенно удобен для сложных моделей вроде Gradient Boosting.

### SHAP limitations / Ограничения SHAP

* может быть вычислительно дорогим;
* интерпретация усложняется при большом количестве признаков;
* при сильно коррелирующих признаках вклады могут распределяться неочевидно;
* `SHAP` объясняет поведение модели, а не истинные причинно-следственные связи в данных.

Итого: `SHAP` - это инструмент для model explanation и анализа feature importance. Он помогает понять, почему модель сделала конкретное предсказание и какие признаки в целом сильнее всего влияют на ее решения.